# Prepared Binding Data for ML Training

In [25]:
import pandas as pd

In [26]:
## Read in Binding Data
cols_to_remove = ['Job_ID', 'smiles', 'iptm']

df = pd.read_csv('combined_50k_filtered.csv').drop(columns=cols_to_remove)

df.head()

,molecular_weight,logP,num_h_donors,num_h_acceptors,num_rotatable_bonds,tpsa,num_rings,num_aromatic_rings,num_atoms,num_heavy_atoms,formal_charge,score,total,vdw,elec,desolv,bsa,num_interacting_residues,neg_log10_affinity_M
0,312.538,7.1127,1,2,19,37.30,0,0,62,22,0,-89.935,-218.023,-48.388,-169.635,-7.620,830.568,22,7.585027
1,301.251,-1.2265,4,8,8,142.52,2,1,36,21,0,-28.529,-31.946,-25.239,-6.707,-1.949,476.808,11,3.301030
2,406.438,4.8204,3,6,11,104.82,4,3,52,30,0,-47.412,-51.750,-45.487,-6.263,-0.672,752.575,19,7.638272
3,520.563,-1.8802,7,9,18,222.13,2,0,71,37,1,-80.642,-197.802,-44.507,-153.295,-5.476,989.693,27,5.191789
4,569.524,-4.5727,7,13,12,253.76,4,1,71,40,0,-29.119,-3.597,-31.511,27.915,-3.190,615.462,18,6.732828


In [27]:
## Remove rows with negative VDW values

df = df[df['vdw'] <= 0]

In [28]:
## Create binding_level class column

binding_levels = {
    "strong": [8, 100], ## <= 0.01uM or pK >= 8
    "medium": [6, 8],   ## 0.01uM < x <= 1uM or 6 <= pK < 8
    "weak": [4, 6],     ## 1uM < x <= 100uM or 4 <= pK < 6
    "none": [0, 4]      ## >100uM or pK < 4
}

df['binding_level'] = df['neg_log10_affinity_M'].apply(lambda x: next((level for level, (low, high) in binding_levels.items() if low <= x < high), 'none'))

df.head()

,molecular_weight,logP,num_h_donors,num_h_acceptors,num_rotatable_bonds,tpsa,num_rings,num_aromatic_rings,num_atoms,num_heavy_atoms,formal_charge,score,total,vdw,elec,desolv,bsa,num_interacting_residues,neg_log10_affinity_M,binding_level
0,312.538,7.1127,1,2,19,37.30,0,0,62,22,0,-89.935,-218.023,-48.388,-169.635,-7.620,830.568,22,7.585027,medium
1,301.251,-1.2265,4,8,8,142.52,2,1,36,21,0,-28.529,-31.946,-25.239,-6.707,-1.949,476.808,11,3.301030,none
2,406.438,4.8204,3,6,11,104.82,4,3,52,30,0,-47.412,-51.750,-45.487,-6.263,-0.672,752.575,19,7.638272,medium
3,520.563,-1.8802,7,9,18,222.13,2,0,71,37,1,-80.642,-197.802,-44.507,-153.295,-5.476,989.693,27,5.191789,weak
4,569.524,-4.5727,7,13,12,253.76,4,1,71,40,0,-29.119,-3.597,-31.511,27.915,-3.190,615.462,18,6.732828,medium


In [29]:
## Check counts of each binding level
df['binding_level'].value_counts()

binding_level
medium    19628
weak      15516
strong     8950
none       2837
Name: count, dtype: int64

In [30]:
## Write to CSV
df.to_csv('combined_50k_filtered_prepped.csv', index=False)